# Step 5: Network Generation

This notebook generates cross-language networks for all video and comment data per dataset, storing the results in gefx files in the output folder specified.

In [ ]:
import pandas as pd
import os
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from itertools import combinations as comb
import networkx as nx
import math
import pprint
from glob import glob
import yaml

In [ ]:
# en = yellow, ja = red, ko = blue, hans = light green, hant = dark green
# multiplecountries = grey
color_dict = {'nan': '#ffffff', 'multi': '#b2beb5', 'en': '#f7b317', 'ja': '#ff3f33', 'ko': '#3380ff', 'zh-hans': '#68f22d', 'zh-hant': '#408c1f'}
color_dict_simplified = {'nan': '#ffffff', 'multi': '#b2beb5', 'en': '#f7b317', 'ja': '#ff3f33', 'ko': '#3380ff', 'zh-hans': '#408c1f', 'zh-hant': '#408c1f'}

dataset_config = "./config/dataset_config.yml"
catfile = '*combined_cleaned_data.csv'
commentsfile = '_comments_cleaned_langdetect.csv'
metadatafile = '_metadata_cleaned_langinfo.csv'
output = './output/'

# change the ingested colums based on the information you want to represent in the network
commentfile_cols = ['commentId', 'videoId', 'videoSearchRegion', 'authorDisplayName', 'authorChannelId', 'lang_lingua', 'updatedAt']
metafile_cols = ['videoId', 'videoSearchRegion', 'defaultLanguage', 'defaultAudioLanguage', 'publishedAt', 
                   'Content match (yes no)','Language Match (yes no)',
                   'YouTube Shorts (yes no)', 'Content Type (choose 0-1)',
                   'Content Focus (choose 0-2)','Values (Subjective Evaluation) muliple possible', 
                   'comp_verdict', 'num_languages', 'language_homogeneity']

In [ ]:
# data screening

print(os.path.abspath(dataset_config))
print(os.path.getsize(dataset_config))
with open(dataset_config, "r") as f:
    config = yaml.safe_load(f)
# Expand paths relative to working dir
directories = {
    key: {
        'wd': (
            [os.path.join(os.getcwd(), path) for path in value['wd']]
            if value['wd'] != None
            else []
        ),
        'catdir': (
            os.path.join(os.getcwd(), value['catdir'])
            if value['catdir'] != None
            else ''
        ),
    }
    for key, value in config.items()
}

pprint.pprint(directories)

In [ ]:
# Function to identify zip files in a directory and create a list of their filenames
def identify_datafiles_in_directory(directory, tp):
    file_list = []
    print('searching for files in:', directory)
    for file_name in os.listdir(directory):
        if file_name.endswith(tp):
            file_list.append(file_name)
    return file_list

In [ ]:
def combineallcommentfiles(files, workd):
    result = pd.DataFrame()
    print(files)
    for file in files:
        #print(file)
        result = pd.concat([result, pd.read_csv(os.path.join(workd, file), sep=',', usecols=commentfile_cols, lineterminator='\n')])
        result['updatedAt']= pd.to_datetime(result['updatedAt'], utc=True)
    print(len(result))
    return result

In [ ]:
def combineallmetadatafiles(files, workd):
    result = pd.DataFrame()
    print(files)
    for file in files:
        print(file)
        tmp_meta = pd.read_csv(os.path.join(workd, file), sep=',', usecols=metafile_cols, lineterminator='\n')
        result = pd.concat([result, tmp_meta])
    print(len(result))
    return result

In [ ]:
def blendcolors(hex):
#    print(hex)
#    print(type(hex))
    if isinstance(hex, list):
        if len(hex) > 2:
            return '#000000'
        elif len(hex) == 2:
            r1, g1, b1 = [int(hex[0][p:p+2], 16) for p in range(1,6,2)]
            r2, g2, b2 = [int(hex[1][p:p+2], 16) for p in range(1,6,2)]
            c = '#{:02x}{:02x}{:02x}'.format((r1+r2) // 2, (g1+g2) //2, (b1+b2)// 2)
            return c
    return hex

In [ ]:
def createnodeinformation(workd, fileending):
    files = identify_datafiles_in_directory(workd, fileending)
    result = combineallmetadatafiles(files, workd)
    result['node_id'] = result['videoId'].copy()
    result = result.drop_duplicates(subset=['videoId', 'videoSearchRegion'], keep="first")

    #print(result.relevanceLanguage.unique())
    #print(color_dict.keys())
    result['lang_color'] = result.apply(lambda row: color_dict[str(row.videoSearchRegion)], axis=1)
    result['lang_color_simple'] = result.apply(lambda row: color_dict_simplified[str(row.videoSearchRegion)], axis=1)

    result = result.groupby('videoId').agg(lambda x: list(set([elem for elem in x if elem == elem])))
    result = result.applymap(lambda x: x[0] if len(x) == 1 else list(x) if len(x) > 1 else 'Unknown')

    result['lang_color'] = result['lang_color'].apply(blendcolors)
    result['lang_color_simple'] = result['lang_color_simple'].apply(blendcolors)

    df = pd.DataFrame(result)
    result_graph = nx.DiGraph()

    # ✅ Clean node attributes here to avoid GEXF export issues
    for node_id, attrs in df.iterrows():
        clean_attrs = {}
        for key, val in attrs.items():
            if isinstance(val, (list, dict)):
                continue  # skip unsupported types
            elif pd.isna(val):
                continue
            else:
                clean_attrs[key] = str(val)  # convert everything else to string
        result_graph.add_node(node_id, **clean_attrs)

    return result_graph


## create relation table between user comments and videos

In [ ]:
def createdirectedmatrix(name, commentdata, workdir, weightime=False):
    G = createnodeinformation(workdir, metadatafile)
    print('node info created')
    for person, group in commentdata.groupby('authorChannelId'):
    # Sort the contributions by timestamp (ascending order)
        group = group.sort_values(by='updatedAt')
        # Step 3: Add edges between projects based on the order of contributions
        for i in range(len(group) - 1):
            # Get the project names and time for consecutive contributions
            project_from = group.iloc[i]['videoId']
            project_to = group.iloc[i + 1]['videoId']
            if weightime:
                timedifference = group.iloc[i+1]['updatedAt'] - group.iloc[i]['updatedAt']
                print(timedifference)
                if timedifference.total_seconds() == 0:
                    timediff = 1
                else:
                    timediff = timedifference.total_seconds()
                weight = 1 / math.ceil(timediff / 3600)
                print(weight)
                # Add directed edge from project_from to project_to
                # The edge direction is based on the time of contribution
            else:
                weight = 1
            if not G.has_edge(project_from, project_to):
                G.add_edge(project_from, project_to, weight=weight)
            else:
                # If the edge already exists, increment the weight (optional)
                G[project_from][project_to]['weight'] += weight
# Step 4: Export the graph to GEXF format for use in Gephi

    nx.write_gexf(G, os.path.join(output, f"ytma_{name}_directed_user_contributions_graph.gexf"))
    
    G_undir = G.to_undirected()
    nx.write_gexf(G_undir, os.path.join(output, f"ytma_{name}_undirected_user_contributions_graph.gexf"))

    # Step 5: View the graph structure (optional)
    print("Nodes:", G.nodes)
    print("Edges:", G.edges)
    return G

# EXECUTE

In [ ]:
for game in directories.keys():
    print(game)
    for dir in directories[game]['wd']:
        print(dir)
        datalabel = dir.split('/')[-2]
        print(datalabel)
        filelist = identify_datafiles_in_directory(dir, commentsfile)
        df_comments = combineallcommentfiles(filelist, dir)
        print(df_comments.head())
        dir_graph = createdirectedmatrix(datalabel, df_comments, dir, False)